# 00 — Telecom localisation feasibility audit

**Outcome:** establish whether the available ONT telemetry, topology and
generator truth can support peer detection and hierarchical localisation.

This is an evidence audit, not a detector. Truth is read only in the final
denominator section and is never joined to observable telemetry values.


## 1. Setup


In [ ]:
import os
import sys
from itertools import combinations
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = Path(
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")
    or ("/content/drive/MyDrive/anomaly_detection" if IN_COLAB
        else Path.home() / "anomaly_detection_data")
).expanduser()
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DATA_ROOT / "research" / "milestone1" if IN_COLAB
    else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
    else Path.cwd() / "notebooks" / "drive_research",
)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import new_output_directory, write_json

SOURCE = Path(os.getenv(
    "TELECOM_SOURCE_ROOT", DATA_ROOT / "telco_syntetic_data"
))
AUDIT_VERSION = "1.0.0"
AUDIT_RUN_ID = os.getenv("TELECOM_AUDIT_RUN_ID", "telecom_localisation_audit_run1")
OUTPUT_ROOT = (
    DATA_ROOT / "outputs" / "audits" / f"v{AUDIT_VERSION}"
    / "telecom" / AUDIT_RUN_ID
)

PANEL_PATH = SOURCE / "reference_dataset.parquet"
TOPOLOGY_PATH = SOURCE / "topology.csv"
FAULT_EVENTS_PATH = SOURCE / "gt_fault_registry.csv"
FAULT_INTERVALS_PATH = SOURCE / "fault_entity_intervals.csv"
for path in (PANEL_PATH, TOPOLOGY_PATH, FAULT_EVENTS_PATH, FAULT_INTERVALS_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

METRICS = [
    "rx_power_dbm", "tx_power_dbm", "temperature_c", "bias_current_ma",
    "voltage_v", "ber", "fec_count", "crc_errors", "uptime_s",
    "reboot_count", "throughput_mbps",
]
TOPOLOGY_LEVELS = {
    "olt_id": ("olt", 0, "physical_topology"),
    "pon_port": ("pon_port", 1, "physical_topology"),
    "splitter_l1": ("splitter_l1", 2, "physical_topology"),
    "splitter_l2": ("splitter_l2", 3, "physical_topology"),
    "geo_cluster": ("geo_cluster", pd.NA, "geographic_context"),
}
SCOPE_MAP = {
    "ont": "entity", "l2": "splitter_l2", "l1": "splitter_l1",
    "pon": "pon_port", "olt": "olt",
}
display(pd.Series({"source": SOURCE, "output": OUTPUT_ROOT}, name="value").to_frame())


## 2. Topology size, completeness and identifiability


In [ ]:
native_topology = pd.read_csv(TOPOLOGY_PATH)
required = {"ont_id", *TOPOLOGY_LEVELS}
missing = required - set(native_topology.columns)
if missing:
    raise ValueError(f"Missing topology fields: {sorted(missing)}")

memberships = []
for native, (group_type, level, family) in TOPOLOGY_LEVELS.items():
    part = native_topology[["ont_id", native]].rename(
        columns={"ont_id": "entity_id", native: "group_id"}
    )
    part["group_type"] = group_type
    part["hierarchy_level"] = level
    part["group_family"] = family
    memberships.append(part)
topology = pd.concat(memberships, ignore_index=True)[[
    "entity_id", "group_type", "group_id", "hierarchy_level", "group_family"
]]
topology[["entity_id", "group_type", "group_id"]] = topology[
    ["entity_id", "group_type", "group_id"]
].astype("string")

if topology[["entity_id", "group_type", "group_id", "group_family"]].isna().any().any():
    raise ValueError("Topology has missing memberships")
if topology.duplicated(["entity_id", "group_type"]).any():
    raise ValueError("An ONT maps to more than one group at the same level")

sizes = (
    topology.groupby(["group_family", "group_type", "group_id"])
    .entity_id.nunique().rename("entities").reset_index()
)
topology_group_audit = (
    sizes.groupby(["group_family", "group_type"])
    .entities.agg(
        groups="size", minimum="min", median="median",
        q90=lambda values: values.quantile(0.90), maximum="max",
    ).reset_index()
)
topology_group_audit["singleton_groups"] = topology_group_audit.group_type.map(
    sizes.loc[sizes.entities.eq(1)].groupby("group_type").size()
).fillna(0).astype(int)
topology_group_audit["missing_memberships"] = 0
display(topology_group_audit)

physical = topology.loc[topology.group_family.eq("physical_topology")]
descendants = {
    key: frozenset(group.entity_id.astype(str))
    for key, group in physical.groupby(["group_type", "group_id"])
}
identical_rows = []
for left, right in combinations(descendants, 2):
    if descendants[left] == descendants[right]:
        identical_rows.append({
            "group_type_left": left[0], "group_id_left": left[1],
            "group_type_right": right[0], "group_id_right": right[1],
            "observable_descendants": len(descendants[left]),
            "identifiability": "same_ont_footprint",
        })
topology_identifiability = pd.DataFrame(identical_rows, columns=[
    "group_type_left", "group_id_left", "group_type_right", "group_id_right",
    "observable_descendants", "identifiability",
])
display(topology_identifiability.head(30))


## 3. Calibration-only effective peer availability


In [ ]:
schema = set(duckdb.sql(f"SELECT * FROM '{PANEL_PATH}' LIMIT 0").df().columns)
available_metrics = [metric for metric in METRICS if metric in schema]
topology_for_sql = topology.loc[
    topology.group_family.eq("physical_topology"),
    ["entity_id", "group_type", "group_id"],
]

with duckdb.connect() as connection:
    connection.register("topology", topology_for_sql)
    first_ts, last_ts = connection.execute(f'''
        SELECT min(CAST(timestamp_utc AS TIMESTAMPTZ)),
               max(CAST(timestamp_utc AS TIMESTAMPTZ))
        FROM '{PANEL_PATH}'
    ''').fetchone()
    calibration_end = first_ts + (last_ts - first_ts) * 0.50
    rows = []
    count_columns = ", ".join(
        f"count(p.{metric}) AS {metric}" for metric in available_metrics
    )
    for group_type in topology_for_sql.group_type.unique():
        # One panel scan per topology level.  Counting one metric at a
        # time is easier to write, but needlessly scans the large source
        # dozens of times.
        group_counts = connection.execute(f'''
            SELECT p.timestamp_utc, t.group_id, {count_columns}
            FROM '{PANEL_PATH}' AS p
            JOIN topology AS t
              ON CAST(p.ont_id AS VARCHAR) = t.entity_id
            WHERE t.group_type = ?
              AND CAST(p.timestamp_utc AS TIMESTAMPTZ) < ?
            GROUP BY p.timestamp_utc, t.group_id
        ''', [group_type, calibration_end]).df()
        for metric in available_metrics:
            counts = group_counts[metric]
            counts = counts.loc[counts.gt(0)]
            peers = counts - 1
            weight = counts.sum()
            rows.append({
                "group_type": group_type,
                "metric_id": metric,
                "group_timestamps": len(counts),
                "valid_entity_observations": int(weight),
                "p10_valid_peers": peers.quantile(0.10),
                "median_valid_peers": peers.median(),
                "proportion_observations_with_7_peers": (
                    counts.loc[peers.ge(7)].sum() / weight if weight else np.nan
                ),
                "proportion_observations_with_15_peers": (
                    counts.loc[peers.ge(15)].sum() / weight if weight else np.nan
                ),
            })
effective_peer_availability = pd.DataFrame(rows)
display(effective_peer_availability)


## 4. Localisation truth and statistical denominators


In [ ]:
registry = pd.read_csv(FAULT_EVENTS_PATH)
intervals = pd.read_csv(FAULT_INTERVALS_PATH)
required_events = {
    "gt_fault_id", "gt_fault_type", "scope", "target",
    "onset_ts", "first_observable_ts", "impact_ts", "repair_ts",
}
if missing := required_events - set(registry.columns):
    raise ValueError(f"Missing fault fields: {sorted(missing)}")

registry["domain_type"] = registry.scope.astype(str).map(SCOPE_MAP)
if registry.domain_type.isna().any():
    raise ValueError(f"Unknown native scopes: {sorted(registry.loc[registry.domain_type.isna(), 'scope'].unique())}")
registry["reference_ts"] = pd.to_datetime(
    registry.first_observable_ts, utc=True, errors="coerce"
).fillna(pd.to_datetime(registry.onset_ts, utc=True, errors="coerce"))
split_edges = [first_ts, calibration_end, first_ts + (last_ts - first_ts) * 0.75, last_ts + pd.Timedelta(seconds=1)]
registry["partition"] = pd.cut(
    registry.reference_ts, bins=split_edges,
    labels=["calibration", "development", "holdout"], right=False,
).astype("string").fillna("unscoreable")

affected = (
    intervals.assign(
        fault_id=intervals.fault_id.astype(str),
        entity_id=intervals.entity_id.astype(str),
    ).groupby("fault_id").entity_id.nunique().rename("affected_entities")
)
truth = registry.assign(fault_id=registry.gt_fault_id.astype(str)).merge(
    affected, on="fault_id", how="left"
)

known_entities = set(topology.entity_id.astype(str))
known_groups = set(zip(physical.group_type.astype(str), physical.group_id.astype(str)))
truth["truth_resolves"] = [
    target in known_entities if domain_type == "entity"
    else (domain_type, target) in known_groups
    for domain_type, target in zip(truth.domain_type, truth.target.astype(str))
]
true_domain_size = []
truth_identifiable = []
for domain_type, target in zip(truth.domain_type, truth.target.astype(str)):
    key = (domain_type, target)
    if domain_type == "entity":
        true_domain_size.append(1)
        truth_identifiable.append(True)
    else:
        members = descendants.get(key, frozenset())
        aliases = [other for other, values in descendants.items() if values == members]
        true_domain_size.append(len(members))
        truth_identifiable.append(len(aliases) == 1)
truth["true_domain_size"] = true_domain_size
truth["truth_identifiable"] = truth_identifiable
truth["affected_fraction"] = truth.affected_entities / truth.true_domain_size.replace(0, np.nan)
truth["fault_scope"] = np.where(truth.affected_entities.gt(1), "multi_entity", "entity_level")

localisation_truth_audit = truth[[
    "fault_id", "gt_fault_type", "partition", "domain_type", "target",
    "affected_entities", "true_domain_size", "affected_fraction",
    "fault_scope", "truth_resolves", "truth_identifiable",
]].rename(columns={"gt_fault_type": "fault_type", "target": "domain_id"})
localisation_denominators = (
    localisation_truth_audit.groupby(
        ["partition", "fault_scope", "domain_type"], dropna=False
    ).agg(
        faults=("fault_id", "nunique"),
        identifiable_faults=("truth_identifiable", "sum"),
        resolved_faults=("truth_resolves", "sum"),
    ).reset_index()
)
display(localisation_denominators)
if not localisation_truth_audit.truth_resolves.all():
    raise ValueError("Some fault locations do not resolve to the supplied topology")


## 5. Freeze the topology capability decision


In [ ]:
level_summary = (
    effective_peer_availability.groupby("group_type")
    .agg(
        median_valid_peers=("median_valid_peers", "median"),
        median_coverage_7=("proportion_observations_with_7_peers", "median"),
    ).reset_index()
)
eligible = set(level_summary.loc[
    level_summary.median_valid_peers.ge(7)
    & level_summary.median_coverage_7.ge(0.80), "group_type"
])
preference = ["splitter_l2", "pon_port", "splitter_l1", "olt"]
primary_peer_level = next((level for level in preference if level in eligible), None)
level_number = {name: level for _, (name, level, family) in TOPOLOGY_LEVELS.items() if family == "physical_topology"}
broader_levels = sorted(
    (level for level in eligible if level_number[level] < level_number[primary_peer_level]),
    key=lambda level: level_number[level], reverse=True,
) if primary_peer_level else []
fallback_peer_level = broader_levels[0] if broader_levels else None

development_shared = localisation_truth_audit.loc[
    localisation_truth_audit.partition.eq("development")
    & localisation_truth_audit.fault_scope.eq("multi_entity")
].fault_id.nunique()
decisions = {
    "audit_version": AUDIT_VERSION,
    "source": str(SOURCE),
    "peer_channel_enabled": primary_peer_level is not None,
    "primary_peer_level": primary_peer_level,
    "fallback_peer_level": fallback_peer_level,
    "minimum_valid_peers": 7,
    "stable_group_scale_from": 15,
    "minimum_group_entities": 3,
    "minimum_group_available_fraction": 0.50,
    "common_mode_levels": sorted(
        topology.loc[topology.group_family.eq("physical_topology"), "group_type"].unique()
    ),
    "development_multi_entity_faults": int(development_shared),
    "localisation_evidence_status": (
        "estimable" if development_shared >= 20 else "descriptive_only_small_denominator"
    ),
    "synthetic_evidence_limit": (
        "validates generator-mechanism recovery, not real Telecom effectiveness"
    ),
}
display(pd.Series(decisions, name="decision").to_frame())

with new_output_directory(OUTPUT_ROOT) as output:
    topology_group_audit.to_csv(output / "topology_group_audit.csv", index=False)
    effective_peer_availability.to_csv(output / "effective_peer_availability.csv", index=False)
    topology_identifiability.to_csv(output / "topology_identifiability.csv", index=False)
    localisation_truth_audit.to_csv(output / "localisation_truth_audit.csv", index=False)
    localisation_denominators.to_csv(output / "localisation_denominators.csv", index=False)
    write_json(output / "topology_decisions.json", decisions)

print("Saved:", OUTPUT_ROOT)
print("Next: 01A_TELECOM_PACK.ipynb")
